In [17]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

def analyze_gpkg_infrastructure(file_path):
    gdf = gpd.read_file(file_path)
    category_counts = gdf['main_category'].value_counts()
    return len(gdf), category_counts

def extract_city_name(filename):
    name = filename.stem.replace('_infrastructure', '')
    return name.replace('_', ' ').title()

def extract_country(city_name):
    country_mapping = {
        'santo domingo': 'Dominican Republic',
        'santiago dominican republic': 'Dominican Republic',
        'havana': 'Cuba',
        'santiago de cuba': 'Cuba',
        'holguín': 'Cuba',
        'port-au-prince': 'Haiti',
        'pétion-ville': 'Haiti',
        'delmas': 'Haiti',
        'castries': 'Saint Lucia',
        'port of spain': 'Trinidad and Tobago',
        'kingston': 'Jamaica',
        'willemstad': 'Curaçao'
    }
    
    city_lower = city_name.lower()
    for city, country in country_mapping.items():
        if city in city_lower:
            return country
    return 'Unknown'

outputs_path = Path("/home/jupyter-daniela/osm_data_retrieval/outputs")
gpkg_files = list(outputs_path.glob('*.gpkg'))

results = []
for gpkg_file in gpkg_files:
    city_name = extract_city_name(gpkg_file)
    country = extract_country(city_name)
    total_features, categories = analyze_gpkg_infrastructure(gpkg_file)
    
    row = {'Ciudad': city_name, 'País': country, 'Total_Features': total_features}
    
    # Add each category as a column
    for category, count in categories.items():
        clean_category = category.replace('_', ' ')
        row[clean_category] = count
    
    results.append(row)

df = pd.DataFrame(results).fillna(0)
df = df.sort_values('Total_Features', ascending=False)

# Reorder columns to show categories first
cols = ['Ciudad', 'País', 'Total_Features']
category_cols = [col for col in df.columns if col not in cols]
df = df[cols + sorted(category_cols)]

print(df.to_string(index=False))

                          Ciudad                País  Total_Features  1 Critical Infrastructure  2 Basic Social Services  3 Mobility Networks  4 Buildings
Santo Domingo Dominican Republic  Dominican Republic          159387                        202                      454               149328         9403
     Santiago Dominican Republic  Dominican Republic           77800                        151                      217                72521         4911
                     Havana Cuba                Cuba           73675                        169                     1049                63154         9303
            Port-Au-Prince Haiti               Haiti           42103                         46                      243                38972         2842
           Santiago De Cuba Cuba                Cuba           20006                         30                      132                14923         4921
                    Holguín Cuba                Cuba           15279  

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import box
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings
warnings.filterwarnings('ignore')

def create_grid(gdf, cell_size=1000):
    bounds = gdf.total_bounds
    minx, miny, maxx, maxy = bounds
    
    grid_cells = []
    x_coords = np.arange(minx, maxx + cell_size, cell_size)
    y_coords = np.arange(miny, maxy + cell_size, cell_size)
    
    for i, x in enumerate(x_coords[:-1]):
        for j, y in enumerate(y_coords[:-1]):
            cell = box(x, y, x + cell_size, y + cell_size)
            grid_cells.append({'geometry': cell, 'grid_id': f'{i}_{j}'})
    
    return gpd.GeoDataFrame(grid_cells, crs=gdf.crs)

def count_features_in_grid(gdf, grid):
    categories = ['1_Critical_Infrastructure', '2_Basic_Social_Services', 
                  '3_Mobility_Networks', '4_Buildings']
    
    results = []
    for _, cell in grid.iterrows():
        intersecting = gdf[gdf.intersects(cell.geometry)]
        row = {'grid_id': cell.grid_id}
        
        for category in categories:
            count = len(intersecting[intersecting['main_category'] == category])
            row[category] = count
        
        results.append(row)
    
    return pd.DataFrame(results)

def create_geographic_heatmap(city_name, gdf, grid_counts, category='4_Buildings'):
    bounds = gdf.total_bounds
    lon_min, lat_min, lon_max, lat_max = bounds
    pad = 0.01
    extent = [lon_min - pad, lon_max + pad, lat_min - pad, lat_max + pad]
    
    fig = plt.figure(figsize=(12, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, color='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.8, color='gray')
    ax.add_feature(cfeature.OCEAN, color='lightblue', alpha=0.5)
    ax.add_feature(cfeature.LAND, color='lightgray', alpha=0.3)
    
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    
    if grid_counts[category].sum() > 0:
        lon_center = gdf.total_bounds[[0, 2]].mean()
        utm_zone = int((lon_center + 180) / 6) + 1
        utm_crs = f'EPSG:{32600 + utm_zone}' if gdf.total_bounds[1] > 0 else f'EPSG:{32700 + utm_zone}'
        
        gdf_utm = gdf.to_crs(utm_crs)
        grid_utm = create_grid(gdf_utm, cell_size=1000)
        grid_latlon = grid_utm.to_crs('EPSG:4326')
        
        grid_with_counts = grid_latlon.merge(
            grid_counts[['grid_id', category]], 
            on='grid_id', 
            how='left'
        ).fillna(0)
        
        data_cells = grid_with_counts[grid_with_counts[category] > 0]
        
        if len(data_cells) > 0:
            data_cells.plot(
                ax=ax, 
                column=category,
                cmap='YlOrRd',
                alpha=0.7,
                transform=ccrs.PlateCarree(),
                legend=True,
                legend_kwds={'label': f'{category.replace("_", " ")} Count', 'shrink': 0.6}
            )
    
    plt.title(f'{city_name}\n{category.replace("_", " ")} Density (1km² grid)', fontsize=14, pad=20)
    return fig

# Create heatmaps for all 4 categories in one figure
santo_domingo = [f for f in gpkg_files if 'santo_domingo' in f.name.lower()][0]
city_name = extract_city_name(santo_domingo)

gdf = gpd.read_file(santo_domingo)
lon_center = gdf.total_bounds[[0, 2]].mean()
utm_zone = int((lon_center + 180) / 6) + 1
utm_crs = f'EPSG:{32600 + utm_zone}' if gdf.total_bounds[1] > 0 else f'EPSG:{32700 + utm_zone}'

gdf_utm = gdf.to_crs(utm_crs)
grid = create_grid(gdf_utm, cell_size=1000)
grid_counts = count_features_in_grid(gdf_utm, grid)

# Create 2x2 subplots for all categories
categories = ['1_Critical_Infrastructure', '2_Basic_Social_Services', '3_Mobility_Networks', '4_Buildings']
fig = plt.figure(figsize=(16, 16))

for i, category in enumerate(categories):
    ax = plt.subplot(2, 2, i+1, projection=ccrs.PlateCarree())
    
    bounds = gdf.total_bounds
    lon_min, lat_min, lon_max, lat_max = bounds
    pad = 0.01
    extent = [lon_min - pad, lon_max + pad, lat_min - pad, lat_max + pad]
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, color='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.8, color='gray')
    ax.add_feature(cfeature.OCEAN, color='lightblue', alpha=0.5)
    ax.add_feature(cfeature.LAND, color='lightgray', alpha=0.3)
    
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    
    if grid_counts[category].sum() > 0:
        grid_utm = create_grid(gdf_utm, cell_size=1000)
        grid_latlon = grid_utm.to_crs('EPSG:4326')
        
        grid_with_counts = grid_latlon.merge(
            grid_counts[['grid_id', category]], 
            on='grid_id', 
            how='left'
        ).fillna(0)
        
        data_cells = grid_with_counts[grid_with_counts[category] > 0]
        
        if len(data_cells) > 0:
            data_cells.plot(
                ax=ax, 
                column=category,
                cmap='YlOrRd',
                alpha=0.7,
                transform=ccrs.PlateCarree(),
                legend=True,
                legend_kwds={'shrink': 0.4}
            )
    
    ax.set_title(f'{category.replace("_", " ")}', fontsize=12, pad=10)

plt.suptitle(f'{city_name}\nInfrastructure Density Heatmaps (1km² grid)', fontsize=16, y=0.98)
plt.tight_layout()
plt.show()